# 03 — Model Evaluation

**AutoClaim AI** | IE University Deep Learning Final Project

Evaluates both trained models on the held-out **test set (374 images)** — data the models never saw during training or validation.

### Summary of results

| Model | Accuracy | Macro F1 | Weighted F1 |
|-------|----------|----------|-------------|
| Custom CNN (from scratch) | 41.4% | 0.374 | 0.387 |
| **MobileNetV2 (transfer)** | **76.5%** | **0.728** | **0.761** |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import json, config
from src.evaluate import evaluate_model, save_metrics_csv, write_evaluation_report
with open(config.CLASS_NAMES_PATH) as f:
    class_names = json.load(f)
print("Classes:", class_names)

## 1. Custom CNN

In [ ]:
r_custom = evaluate_model(config.CUSTOM_MODEL_PATH, "custom_cnn", class_names)
if r_custom:
    print("Custom CNN test accuracy:", r_custom["accuracy"])

### Custom CNN — per-class results

| Class | Precision | Recall | F1 | Support |
|-------|-----------|--------|----|---------|
| crack | 0.00 | 0.00 | 0.00 | 8 |
| dent | 0.36 | 0.22 | 0.27 | 109 |
| glass shatter | 0.70 | 0.85 | 0.76 | 65 |
| lamp broken | 0.23 | 0.76 | 0.35 | 42 |
| scratch | 0.51 | 0.20 | 0.28 | 122 |
| tire flat | 0.48 | 0.71 | 0.57 | 28 |

**Observations:**
- `glass shatter` (F1=0.76) and `tire flat` (F1=0.57) are the strongest — visually distinctive patterns
- `crack` (F1=0.00) — model never predicts this class; only 8 test samples and 61 train samples
- `lamp broken` shows high recall (76%) but low precision — model over-predicts this class

## 2. Transfer Learning CNN — MobileNetV2

> **BEYOND CLASS MATERIAL**

In [ ]:
r_transfer = evaluate_model(config.TRANSFER_MODEL_PATH, "transfer_mobilenetv2", class_names)
if r_transfer:
    print("Transfer test accuracy:", r_transfer["accuracy"])

### MobileNetV2 — per-class results

| Class | Precision | Recall | F1 | Support |
|-------|-----------|--------|----|---------|
| crack | 0.67 | 0.25 | 0.36 | 8 |
| dent | 0.70 | 0.74 | 0.72 | 109 |
| **glass shatter** | **0.90** | **0.98** | **0.94** | 65 |
| lamp broken | 0.58 | 0.79 | 0.67 | 42 |
| scratch | 0.79 | 0.65 | 0.71 | 122 |
| **tire flat** | **0.96** | **0.96** | **0.96** | 28 |

**Observations:**
- `tire flat` (F1=0.96) and `glass shatter` (F1=0.94) are near-perfect — safety-critical classes well detected
- `crack` (F1=0.36) still low — fundamental data limitation (only 8 test / 61 train images), not a model failure
- All classes improved significantly vs the custom CNN

## 3. Summary Table

In [ ]:
results = [r for r in [r_custom, r_transfer] if r is not None]
if results:
    df = save_metrics_csv(results)
    write_evaluation_report(results, class_names)
    print(df.to_string(index=False))

## 4. Training Curves

In [ ]:
import json, matplotlib.pyplot as plt

for name in ["custom", "transfer"]:
    path = os.path.join(config.METRICS_DIR, f"history_{name}.json")
    if not os.path.exists(path):
        continue
    with open(path) as f:
        hist = json.load(f)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(hist["loss"],     label="train"); a1.plot(hist["val_loss"],     label="val")
    a1.set_title(f"{name} — Loss");     a1.legend(); a1.grid(alpha=.3)
    a2.plot(hist["accuracy"], label="train"); a2.plot(hist["val_accuracy"], label="val")
    a2.set_title(f"{name} — Accuracy"); a2.legend(); a2.grid(alpha=.3)
    plt.suptitle(f"Training Curves — {name}"); plt.tight_layout(); plt.show()

## 5. Business Interpretation

| Damage | Safety risk | Custom CNN F1 | MobileNetV2 F1 | Triage path |
|--------|-------------|:---:|:---:|------|
| scratch | Low | 0.28 | 0.71 | Fast-track if conf ≥ 80% |
| dent | Low | 0.27 | 0.72 | Fast-track if conf ≥ 80% |
| lamp broken | Medium | 0.35 | 0.67 | Priority if conf ≥ 70% |
| glass shatter | High (safety) | 0.76 | **0.94** | Priority if conf ≥ 70% |
| tire flat | High (safety) | 0.57 | **0.96** | Priority if conf ≥ 70% |
| crack | High (structural) | 0.00 | 0.36 | Human Review (data-limited) |

**Key finding**: The triage confidence thresholds in `config.py` route low-confidence or severe-class predictions to human review, catching the cases the model is uncertain about — including crack, where the model signals uncertainty by returning low confidence.